In [1]:
import pandas as pd
import tqdm, os
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../../..")

from utils.information_theoretic_calculation import InformationTheoreticCalculator
from utils.data_processing import format_context_with_boundaries
from utils.ipa_processing import syllabify_ipa
SEP = "_"

In [2]:
df = pd.read_csv("../dataset_construction/datasets/processed_beus.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])
df = format_context_with_boundaries(df, lang="be")

In [3]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,ac_f,art_f,subject
0,1.47,1.53,1.53,1.60,p,l,p,please,00me_035_xaud_frames.npy_0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
1,1.53,1.60,1.60,1.75,l,i,p_l,please,00me_035_xaud_frames.npy_0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
2,1.60,1.75,1.75,1.83,i,z,p_l_i,please,00me_035_xaud_frames.npy_0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
3,1.83,1.97,1.97,2.16,k,ɔ,k,call,00me_035_xaud_frames.npy_1,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
4,1.97,2.16,2.16,2.17,ɔ,l,k_ɔ,call,00me_035_xaud_frames.npy_1,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,xaud
...,...,...,...,...,...,...,...,...,...,...,...,...
163314,2.23,2.32,2.32,2.39,h,i,h,he,80me_222_aud_frames.npy_6,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,aud
163315,2.39,2.53,2.53,2.67,f,ɛ,f,fairly,80me_222_aud_frames.npy_7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,aud
163316,2.53,2.67,2.67,2.73,ɛ,l,f_ɛ_.,fairly,80me_222_aud_frames.npy_7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,aud
163317,2.67,2.73,2.73,2.79,l,i,f_ɛ_._l,fairly,80me_222_aud_frames.npy_7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,aud


In [4]:
with open("../../pronunciation_counts/pronunciation_counts_be.pkl", "rb") as f:
    pronunciation_counts = pickle.load(f)
pronunciation_counts = {syllabify_ipa(k, lang="be").replace(".", "_._"): v for k, v in pronunciation_counts.items() if "*" not in k and k != ""}

In [5]:
bigrams = set()
for phone1, phone2 in tqdm.tqdm(zip(df["phone1"], df["phone2"]), total=len(df)):
    bigrams.add(phone1+SEP+phone2)

all_phones = set()
for pronunciation in tqdm.tqdm(pronunciation_counts):

    for phone in pronunciation.split(SEP):
        all_phones.add(phone)

monogram_counts = {}
for pronunciation, count in tqdm.tqdm(pronunciation_counts.items()):
    phones = pronunciation.split(SEP)
    for phone in phones:
        if phone not in monogram_counts:
            monogram_counts[phone] = 0
        monogram_counts[phone] += count

100%|██████████| 554222/554222 [00:00<00:00, 599309.87it/s]


In [6]:
calculator = InformationTheoreticCalculator(
    pronunciation_counts=pronunciation_counts,
    phone_counts=monogram_counts,
    sep=SEP
)

100%|██████████| 554222/554222 [00:06<00:00, 91289.77it/s] 


In [7]:
data = {
    "bigram_surprisal": [],
    "phone2_pos_entropy": [],
}

In [8]:
data["bigram_surprisal"] = df.apply(lambda x: calculator.get_contextual_surprisal(SEP.join(x["context"].split(SEP)[:-1]), x["phone1"] + SEP + x["phone2"]), axis=1).tolist()
data["phone2_pos_entropy"] = df.apply(lambda x: calculator.get_target_pos_entropy(x["context"]), axis=1).tolist()
data["phone1_freq"] = df.apply(lambda x: monogram_counts.get(x["phone1"], None), axis=1).tolist()
data["phone2_freq"] = df.apply(lambda x: monogram_counts.get(x["phone2"], None), axis=1).tolist()

In [9]:
pd.DataFrame(data).to_csv("surprisal_beus.csv", index=False)